# Численное моделирование распространения сейсмических волн в двумерной среде MILEN SEM 2D. Часть вторая.

## Глава I: Модель для геофизиков

### Научная задача

Наше исследование заключается в построении конечноэлементной и конечноразностной моделей с последующим совместным их решением.

Мы уже построили конечноэлементную модель для программы CAE-Fidesys
Теперь нужно построить аналогичную конечно-разностную двумерную модель задачи в файле формата, совместимого с tesseral.

In [1]:
import numpy as np
import segyio
from pathlib import Path

#### Задача 1: Анализ формата данных Tesseral ####

In [2]:
# Анализируем пример из Tesseral
example_vp_path = 'src/tesseral_example/Vp_model.sgy'

print("="*60)
print("АНАЛИЗ ФОРМАТА SEG-Y ИЗ TESSERAL")
print("="*60)

with segyio.open(example_vp_path, 'r', ignore_geometry=True) as f:
    print(f"\nФайл: {example_vp_path}")
    print(f"Количество трасс: {f.tracecount}")
    print(f"Количество сэмплов на трассу: {len(f.trace[0])}")
    print(f"Формат данных: {f.format}")
    
    # Читаем заголовок первой трассы
    print(f"\nЗаголовок первой трассы:")
    header = f.header[0]
    print(f"  Inline: {header[segyio.TraceField.INLINE_3D]}")
    print(f"  Crossline: {header[segyio.TraceField.CROSSLINE_3D]}")
    print(f"  CDP X: {header[segyio.TraceField.CDP_X]}")
    print(f"  CDP Y: {header[segyio.TraceField.CDP_Y]}")
    print(f"  Sample interval: {header[segyio.TraceField.TRACE_SAMPLE_INTERVAL]}")
    
    # Читаем бинарный заголовок
    print(f"\nБинарный заголовок:")
    bin_header = f.bin
    print(f"  Job ID: {bin_header[segyio.BinField.JobID]}")
    print(f"  Samples: {bin_header[segyio.BinField.Samples]}")
    print(f"  Interval: {bin_header[segyio.BinField.Interval]}")
    print(f"  Format: {bin_header[segyio.BinField.Format]}")
    
    # Читаем данные первой трассы
    trace_0 = f.trace[0]
    print(f"\nДанные первой трассы:")
    print(f"  Минимум: {trace_0.min():.2f}")
    print(f"  Максимум: {trace_0.max():.2f}")
    print(f"  Среднее: {trace_0.mean():.2f}")
    print(f"  Первые 5 значений: {trace_0[:5]}")

# Анализируем все три файла
for param_name in ['Vp', 'Vs', 'Density']:
    param_path = f'src/tesseral_example/{param_name}_model.sgy'
    with segyio.open(param_path, 'r', ignore_geometry=True) as f:
        data = segyio.tools.collect(f.trace[:])
        print(f"\n{param_name}:")
        print(f"  Форма данных: {data.shape}")
        print(f"  Диапазон: [{data.min():.2f}, {data.max():.2f}]")

АНАЛИЗ ФОРМАТА SEG-Y ИЗ TESSERAL

Файл: src/tesseral_example/Vp_model.sgy
Количество трасс: 60501
Количество сэмплов на трассу: 1501
Формат данных: 4-byte IEEE float

Заголовок первой трассы:
  Inline: 2303
  Crossline: 5903
  CDP X: 66116302
  CDP Y: 129377889
  Sample interval: 2000

Бинарный заголовок:
  Job ID: 1
  Samples: 1501
  Interval: 2000
  Format: 5

Данные первой трассы:
  Минимум: 1514.39
  Максимум: 5139.81
  Среднее: 3118.19
  Первые 5 значений: [1844.7136 1849.0387 1849.0664 1830.371  1817.602 ]

Vp:
  Форма данных: (60501, 1501)
  Диапазон: [1428.60, 5139.81]

Vs:
  Форма данных: (60501, 1501)
  Диапазон: [784.88, 2826.04]

Density:
  Форма данных: (60501, 1501)
  Диапазон: [1.86, 2.55]


#### Задача 2: Подготовка и выгрузка наших данных ####

In [3]:
print("\n" + "="*60)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ TESSERAL")
print("="*60)

# Загружаем наши декартовы данные
data_path = Path('data/dev_1_7_material_grids.npz')
data = np.load(data_path)

material_grid = data['material_grid']      # [2350, 550, 3]
coords_grid = data['coords_grid']          # [2350, 550, 2]
layer_indexes_grid = data['layer_indexes_grid']  # [2350, 550]

print(f"\nЗагружены данные из {data_path}")
print(f"Размер сетки материала: {material_grid.shape}")
print(f"Размер сетки координат: {coords_grid.shape}")

# Извлекаем свойства
E_grid = material_grid[:, :, 0]   # Модуль Юнга (Па)
nu_grid = material_grid[:, :, 1]  # Коэффициент Пуассона
rho_grid = material_grid[:, :, 2] # Плотность (кг/м³)

# Вычисляем скорости волн
Vp_grid = np.sqrt(E_grid / rho_grid * (1 - nu_grid) / 
                  ((1 + nu_grid) * (1 - 2*nu_grid)))
Vs_grid = np.sqrt(E_grid / (2 * rho_grid * (1 + nu_grid)))

# Плотность в г/см³ (стандарт для геофизики)
density_grid = rho_grid / 1000.0

print(f"\nДиапазоны параметров:")
print(f"  Vp: {Vp_grid.min():.1f} - {Vp_grid.max():.1f} м/с")
print(f"  Vs: {Vs_grid.min():.1f} - {Vs_grid.max():.1f} м/с")
print(f"  Плотность: {density_grid.min():.3f} - {density_grid.max():.3f} г/см³")

# Получаем параметры сетки
x_coords = coords_grid[:, 0, 0]  # Координаты X
y_coords = coords_grid[0, :, 1]  # Координаты Y (глубины)

dx = x_coords[1] - x_coords[0] if len(x_coords) > 1 else 5.0
dy = y_coords[1] - y_coords[0] if len(y_coords) > 1 else 5.0

print(f"\nПараметры сетки:")
print(f"  Nx = {len(x_coords)}, Ny = {len(y_coords)}")
print(f"  dx = {dx:.1f} м, dy = {dy:.1f} м")
print(f"  X: {x_coords[0]:.1f} - {x_coords[-1]:.1f} м")
print(f"  Y: {y_coords[0]:.1f} - {y_coords[-1]:.1f} м")


ПОДГОТОВКА ДАННЫХ ДЛЯ TESSERAL

Загружены данные из data/dev_1_7_material_grids.npz
Размер сетки материала: (2350, 550, 3)
Размер сетки координат: (2350, 550, 2)

Диапазоны параметров:
  Vp: 1209.4 - 5267.8 м/с
  Vs: 664.3 - 2896.5 м/с
  Плотность: 1.814 - 2.578 г/см³

Параметры сетки:
  Nx = 2350, Ny = 550
  dx = 5.0 м, dy = 5.0 м
  X: 2.5 - 11747.5 м
  Y: 2.5 - 2747.5 м


#### Функция для записи данных в формат SEG-Y ####

In [4]:
def write_segy_2d(filename, data, dx=5.0, dy=5.0, x_origin=0.0, y_origin=0.0):
    """
    Записывает 2D массив данных в формат SEG-Y.
    
    Args:
        filename: имя выходного файла
        data: 2D массив данных [nx, ny]
        dx: шаг по X (м)
        dy: шаг по Y (м)  
        x_origin: начало координат по X (м)
        y_origin: начало координат по Y (м)
    """
    nx, ny = data.shape
    
    # Создаем спецификацию SEG-Y
    spec = segyio.spec()
    spec.format = 5  # IEEE float
    spec.samples = range(ny)  # индексы сэмплов
    spec.tracecount = nx  # количество трасс = количество точек по X
    
    # Интервал в микросекундах (для совместимости с сейсмикой)
    # Используем dy в метрах, умножаем на 1000 для перевода в мкс
    spec.interval = int(dy * 1000)
    
    with segyio.create(filename, spec) as f:
        # Записываем бинарный заголовок
        f.bin = {
            segyio.BinField.JobID: 1,
            segyio.BinField.Samples: ny,
            segyio.BinField.Interval: spec.interval,
            segyio.BinField.Format: 5  # IEEE float
        }
        
        # Записываем трассы
        for i in range(nx):
            # Заголовок трассы
            f.header[i] = {
                segyio.TraceField.TRACE_SEQUENCE_FILE: i + 1,
                segyio.TraceField.TRACE_SEQUENCE_LINE: i + 1,
                segyio.TraceField.INLINE_3D: i + 1,
                segyio.TraceField.CROSSLINE_3D: 1,
                segyio.TraceField.CDP_X: int(x_origin + i * dx),
                segyio.TraceField.CDP_Y: int(y_origin),
                segyio.TraceField.TRACE_SAMPLE_COUNT: ny,
                segyio.TraceField.TRACE_SAMPLE_INTERVAL: spec.interval
            }
            
            # Данные трассы (столбец по Y)
            f.trace[i] = data[i, :].astype(np.float32)
    
    print(f"  Записан файл: {filename}")
    print(f"    Размер: {nx} × {ny}")
    print(f"    Диапазон: [{data.min():.2f}, {data.max():.2f}]")

#### Запись данных в файлы SEG-Y ####

In [5]:
print("\n" + "="*60)
print("ЗАПИСЬ ДАННЫХ В ФОРМАТ SEG-Y")
print("="*60 + "\n")

output_dir = Path('data')

# Записываем Vp
vp_output = output_dir / 'dev_2_1_Vp_model.sgy'
write_segy_2d(str(vp_output), Vp_grid, dx=dx, dy=dy)

# Записываем Vs
vs_output = output_dir / 'dev_2_1_Vs_model.sgy'
write_segy_2d(str(vs_output), Vs_grid, dx=dx, dy=dy)

# Записываем плотность
density_output = output_dir / 'dev_2_1_Density_model.sgy'
write_segy_2d(str(density_output), density_grid, dx=dx, dy=dy)

print("\n" + "="*60)
print("ПРОВЕРКА ЗАПИСАННЫХ ФАЙЛОВ")
print("="*60)

# Проверяем записанные файлы
for param_name, output_file in [('Vp', vp_output), ('Vs', vs_output), ('Density', density_output)]:
    with segyio.open(str(output_file), 'r', ignore_geometry=True) as f:
        data_check = segyio.tools.collect(f.trace[:])
        print(f"\n{param_name} (проверка):")
        print(f"  Файл: {output_file.name}")
        print(f"  Форма: {data_check.shape}")
        print(f"  Диапазон: [{data_check.min():.2f}, {data_check.max():.2f}]")
        print(f"  Количество трасс: {f.tracecount}")
        print(f"  Сэмплов на трассу: {len(f.trace[0])}")

print("\n" + "="*60)
print("ЗАВЕРШЕНО")
print("="*60)


ЗАПИСЬ ДАННЫХ В ФОРМАТ SEG-Y

  Записан файл: data/dev_2_1_Vp_model.sgy
    Размер: 2350 × 550
    Диапазон: [1209.41, 5267.80]
  Записан файл: data/dev_2_1_Vs_model.sgy
    Размер: 2350 × 550
    Диапазон: [664.32, 2896.45]
  Записан файл: data/dev_2_1_Density_model.sgy
    Размер: 2350 × 550
    Диапазон: [1.81, 2.58]

ПРОВЕРКА ЗАПИСАННЫХ ФАЙЛОВ

Vp (проверка):
  Файл: dev_2_1_Vp_model.sgy
  Форма: (2350, 550)
  Диапазон: [1209.41, 5267.80]
  Количество трасс: 2350
  Сэмплов на трассу: 550

Vs (проверка):
  Файл: dev_2_1_Vs_model.sgy
  Форма: (2350, 550)
  Диапазон: [664.32, 2896.45]
  Количество трасс: 2350
  Сэмплов на трассу: 550

Density (проверка):
  Файл: dev_2_1_Density_model.sgy
  Форма: (2350, 550)
  Диапазон: [1.81, 2.58]
  Количество трасс: 2350
  Сэмплов на трассу: 550

ЗАВЕРШЕНО


### Выводы

В Главе VIII была выполнена подготовка данных для конечно-разностного моделирования в Tesseral:

1. **Анализ формата:** Изучена структура файлов SEG-Y из примера Tesseral
2. **Конвертация данных:** Преобразованы декартовы данные материала в формат SEG-Y
3. **Выходные файлы:** Созданы три файла для Tesseral:
   - data/dev_1_8_Vp_model.sgy - модель скоростей продольных волн
   - data/dev_1_8_Vs_model.sgy - модель скоростей поперечных волн
   - data/dev_1_8_Density_model.sgy - модель плотности

Данные готовы для использования в программе Tesseral для конечно-разностного моделирования.